# Follow-up experiments: retention and multi-seed confirmation

Open this notebook in VS Code with your existing Kaggle Jupyter kernel. Enable
GPU and Internet. Attach **phase3_results.zip + phase4_results.zip**, their complete
extracted contents, or **CrackLab_Phase8_v1.0.0.zip**. No Phase 8 files are overwritten.
Original datasets are restored from pinned commits and checked against all 657 pairs.

Default run: six validation-only retention conditions per original parent, then
the locked no-replay/replay/selected matrix (only if a distillation candidate passes).
All old-query results remain exploratory. A no-candidate result is a valid failure.
New independent scenes and timed human corrections cannot be generated by this notebook.

Run the backup cell after completion or a safe interruption and download its ZIP
and SHA256 before ending a Kaggle session. Do not upgrade PyTorch mid-run; exact
resume requires the same code/configuration and compatible runtime.


In [1]:
payload='...'
expected='d607572a1eead1f063e187165813574718631bf7c0b2dfc6d09753f4cc07c586'
specs=[{'phase': 3, 'model': 'litecrack', 'archive': 'phase3_results.zip', 'archive_sha256': 'd1d9c4fa7c42efb64e2f30f775fbbf8f5973f22676f591ba1537523d63119470', 'pointer': 'artifacts/phase3/litecrack/checkpoints/best.json', 'pointer_sha256': '2dfdcca8668156f7647238ae34e971b7552b65a6be74b9aceb66f163b225512b', 'checkpoint': 'artifacts/phase3/litecrack/checkpoints/step-000002970-92069633.pt', 'checkpoint_sha256': '9ac52e68c86938f826ddf9bf5176ce8abba830c4ecb3cdd01dd35823777c1fa2'}, {'phase': 4, 'model': 'litecrack_ba', 'archive': 'phase4_results.zip', 'archive_sha256': '0851dc03b170e41df2639432a2373579e594a7398f8e0c78183eb5f2a86d652c', 'pointer': 'artifacts/phase4/litecrack_ba/checkpoints/best.json', 'pointer_sha256': '6a35e1ae9fac9cd31ca5c4ed2cc03e29c3bc5dd87f89d7658374446963e5521c', 'checkpoint': 'artifacts/phase4/litecrack_ba/checkpoints/step-000002580-2f498b1b.pt', 'checkpoint_sha256': '786fbf654af8ff1b0e139328ac5d4f121ce9c215dd504988054ae21502f661f6'}]
import os,sys,json,hashlib,base64,zlib,subprocess,zipfile,shutil,signal
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
raw=zlib.decompress(base64.b64decode(payload));assert hashlib.sha256(raw).hexdigest()==expected
files=json.loads(raw)
project=Path('/kaggle/working/road_crack_followup_v1');project.mkdir(exist_ok=True)
for name,text in files.items():
    path=project/name;path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():assert path.read_text()==text,'Existing code differs; preserve it and use a new protocol folder'
    else:path.write_text(text)
os.chdir(project)
if str(project) not in sys.path:sys.path.insert(0,str(project))
parent_root=project/'parents';parent_root.mkdir(exist_ok=True)
search_roots=[Path('/kaggle/working'),Path('/kaggle/input')]
for spec in specs:
    destination=parent_root/spec['archive']
    if destination.exists():assert hashlib.sha256(destination.read_bytes()).hexdigest()==spec['archive_sha256'];continue
    archives=[r/spec['archive'] for r in search_roots if (r/spec['archive']).is_file()]
    archives+=list(Path('/kaggle/input').rglob(spec['archive']))
    if archives:
        assert hashlib.sha256(archives[0].read_bytes()).hexdigest()==spec['archive_sha256'],'Wrong original archive'
        shutil.copyfile(archives[0],destination);continue
    releases=list(Path('/kaggle/input').rglob('CrackLab_Phase8_v1.0.0.zip'))
    if releases:
        with zipfile.ZipFile(releases[0]) as z:content=z.read(spec['archive'])
        assert hashlib.sha256(content).hexdigest()==spec['archive_sha256'];destination.write_bytes(content);continue
    candidates=list(Path('/kaggle/input').rglob('best.json'))
    candidates+=[r/spec['pointer'] for r in search_roots if (r/spec['pointer']).exists()]
    found=None
    for pointer in candidates:
        if pointer.as_posix().endswith('/'+spec['pointer']) and hashlib.sha256(pointer.read_bytes()).hexdigest()==spec['pointer_sha256']:
            checkpoint=pointer.parent/Path(spec['checkpoint']).name
            if checkpoint.is_file() and hashlib.sha256(checkpoint.read_bytes()).hexdigest()==spec['checkpoint_sha256']:
                found=(pointer,checkpoint);break
    assert found,'Attach '+spec['archive']+' or complete extracted original checkpoint contents'
    for source,name in zip(found,(spec['pointer'],spec['checkpoint'])):
        target=parent_root/name;target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():assert target.read_bytes()==source.read_bytes()
        else:shutil.copyfile(source,target)
import torch
assert torch.cuda.is_available(),'GPU required: enable the Kaggle GPU and reconnect the existing Jupyter kernel'
print('GPU:',torch.cuda.get_device_name(0),'PyTorch:',torch.__version__)
subprocess.run([sys.executable,'scripts/restore_datasets.py','--manifest','data/manifests/pilot_v1.json',
    '--destination','/kaggle/working/road_crack_followup_data','--sources','configs/data_sources.json',
    '--report','dataset_restore.json'],check=True)
data_root=Path(json.loads((project/'dataset_restore.json').read_text())['data_root'])
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=open(project/'gpu-environment.lock.txt','w'),check=True)


GPU: Tesla T4 PyTorch: 2.11.0+cu128
Previous data unavailable/incomplete. Restoring pinned repositories to: /kaggle/working/road_crack_followup_data


HEAD is now at 8202a60 Update README.md
HEAD is now at bd8fbd7 Add files via upload


Restoration complete: verified 657 pairs against the unchanged frozen manifest


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'freeze'], returncode=0)

In [2]:
import xml.etree.ElementTree as ET
test_dir=project/'test_evidence';test_dir.mkdir(exist_ok=True)
result=subprocess.run([sys.executable,'-m','pytest','tests/test_followup.py','-q','-rA',
    '-p','no:cacheprovider','--basetemp='+str(test_dir/'tmp'),
    '--junitxml='+str(test_dir/'tests.xml')],text=True,capture_output=True)
(test_dir/'tests.log').write_text(result.stdout+result.stderr)
print(result.stdout,result.stderr);assert result.returncode==0
recovery=[c for c in ET.parse(test_dir/'tests.xml').findall('.//testcase') if 'test_followup_exact_resume' in c.attrib.get('name','')]
assert len(recovery)==4 and all(c.find('skipped') is None for c in recovery),'Both CPU and CUDA resume must pass'


..................                                                       [100%]
==================================== PASSES ====================================
________________ test_validation_only_search_and_protocol_lock _________________
----------------------------- Captured stdout call -----------------------------
ba no_replay {"passed": false, "source_change_points": -0.011745840311050415, "target_gain_points": -0.01240074634552002}
ba replay {"passed": true, "source_change_points": 0.033395737409591675, "target_gain_points": 0.07930199305216656}
ba distill {"passed": true, "source_change_points": 0.033395737409591675, "target_gain_points": 0.07930199305216656}
_____________ test_confirmation_matrix_finishes_before_test_scores _____________
----------------------------- Captured stdout call -----------------------------
ba no_replay {"passed": false, "source_change_points": -0.011745840311050415, "target_gain_points": -0.01240074634552002}
ba replay {"passed": true, "source_cha

In [3]:
def run_followup(arguments):
    command=[sys.executable,'scripts/run_followup.py','--data-root',str(data_root),'--archive-root',str(parent_root)]+arguments
    with open(project/'execution.log','a',buffering=1) as log:
        worker=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in worker.stdout:print(line,end='',flush=True);log.write(line)
            code=worker.wait()
        except KeyboardInterrupt:
            worker.send_signal(signal.SIGTERM);worker.wait(timeout=120);raise
    assert code==0,'Study failed; preserve logs/checkpoints before retrying'
run_followup(['--stage','preflight','--output',str(project/'readiness')])
assert shutil.disk_usage(project).free>4*1024**3,'Need free space for immutable adaptation snapshots'
development=project/'retention_development'
run_followup(['--stage','develop','--output',str(development)])
assert (development/'selection.json').exists(),'Safe interruption: rerun resumes, then export recovery'
print((development/'selection.json').read_text())


{
  "cuda_available": true,
  "torch": "2.11.0+cu128",
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "manifest_matches": true,
  "counts": {
    "source_train": 240,
    "source_validation": 60,
    "source_holdout": 237,
    "target_acquisition": 25,
    "target_query": 95
  },
  "verified_pairs": {
    "source_train": 240,
    "source_validation": 60,
    "source_holdout": 237,
    "target_acquisition": 25,
    "target_query": 95
  },
  "missing_files": 0,
  "mismatched_files": 0,
  "missing_examples": [],
  "mismatch_examples": [],
  "free_disk_gib": 19.04,
  "ready_for_gpu_retraining": true,
  "independent_scene_data": "Must be supplied with complete labels and physical-site provenance.",
  "human_effort": "Requires real timed annotation sessions; not derived from imported labels."
}
litecrack no_replay {"passed": false, "source_change_points": -5.439723888412118, "target_gain_points": 18.850082037970424}
litecrack replay {"passed": true, "source_change_points": -1.5104650

In [4]:
confirmation=project/'retention_matrix'
run_followup(['--stage','confirm','--development',str(development),'--output',str(confirmation)])
if (confirmation/'results.json').exists():
    subprocess.run([sys.executable,'scripts/summarize_followup.py','--results',str(confirmation/'results.json'),
        '--output',str(confirmation/'summary.json')],check=True)
print((confirmation/'status.json').read_text() if (confirmation/'status.json').exists() else 'Stopped safely; export backup')


{
  "adaptations": 90,
  "state": "complete",
  "unmet_models": []
}



## Separate multi-seed source retraining (GPU, substantial storage/time)

This cell is opt-in and does not claim the three-seed work ran by default. Run one
seed block at a time, exporting checkpoints between sessions. Latest/best snapshots
are backed up; originals are never pruned automatically. Restore the same project
directory and runtime for exact resume. The completed parent index requires all
three seeds and three models. Source checkpoints use the original 30-step/epoch
cadence; adaptation snapshots are saved every step.


In [5]:
RUN_SOURCE_RETRAINING=False
TRAINING_SEED=42 # Run 42, 73, 101 with all previous blocks restored; do not select the best seed.
if RUN_SOURCE_RETRAINING:
    assert shutil.disk_usage(project).free>8*1024**3,'Need headroom; preserve/export older snapshots, never delete blindly'
    source=project/'source_multiseed'
    run_followup(['--stage','source-train','--only-seed',str(TRAINING_SEED),'--output',str(source)])
    index=source/'parents.json'
    if json.loads(index.read_text()).get('complete'):
        run_followup(['--stage','source-evaluate','--parents-index',str(index),'--output',str(project/'source_multiseed_evaluation')])
        multi_dev=project/'multiseed_development'
        run_followup(['--stage','develop','--parents-index',str(index),'--output',str(multi_dev)])
        run_followup(['--stage','confirm','--parents-index',str(index),'--development',str(multi_dev),'--output',str(project/'multiseed_matrix')])
else:print('Multi-seed retraining not executed. Enable explicitly when GPU/storage are ready.')


Multi-seed retraining not executed. Enable explicitly when GPU/storage are ready.


In [6]:
from datetime import datetime,timezone
backup=Path('/kaggle/working')/('followup_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.zip')
subprocess.run([sys.executable,'scripts/package_followup.py','--root',str(project),'--output',str(backup)],check=True)
print('Download ZIP and SHA256 through Kaggle Files:',backup)


{
  "archive": "/kaggle/working/followup_20261101T121528630671Z.zip",
  "sha256": "759807ddd924fe8762e39a261b459f6e796d42688d7fe5fbe46e6e943234322c",
  "files": 1083,
  "bytes": 247195791
}
Download ZIP and SHA256 through Kaggle Files: /kaggle/working/followup_20261101T121528630671Z.zip
